## Задача Коши

Дана задача Коши вида

$$\begin{gathered}u'(x) = f(x, u(x)), \quad x \in [x_0, X], \\ u(x_0) = u_0, \end{gathered}$$

где  
$$\begin{gathered}[x_0, X] = [0.35, 1.35], \\ f(x, u(x)) = \frac{u(x)}{x} + 0.35 x e^x - 0.65 x \sin x, \\ u_0 = 0.35(0.35e^{0.35} + 0.65\cos 0.35) \approx 0.387543 \end{gathered}$$

## Задания

1. Найти приближенное решение задачи Коши на сетке узлов при 10-ти разбиениях отрезка интегрирования, применяя методы 1-го и 2-го порядков точности.

- Неявный метод Эйлера с реализацией по методу простых итераций;
- Явный метод Рунге-Кутта 2-го порядка с $\alpha=1$;
- Неявный метод Адамса 2-го порядка с реализацией по методу простых итераций.

2. Используя таблицу результатов, получить погрешности методов, сравнивая приближенное решение с точным ($u(x) = x f(x)$).

3. Оценивая величину истинной погрешности, сделать вывод о точности каждого используемого метода.
```

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def func(x, y):
    return y/x + 0.35 * x * np.exp(x) - 0.65 * x * np.sin(x)

num_nodes = 11
x = np.linspace(0.35, 1.35, num_nodes)
u_0 = 0.35 * (0.35 * np.exp(0.35) + 0.65 * np.cos(0.35))
x, u_0

(array([0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.95, 1.05, 1.15, 1.25, 1.35]),
 np.float64(0.38754306687545265))

### 1. Неявный метод Эйлера с реализацией по методу простых итераций

$$y_{j+1}=y_j+hf(x_{j+1}, y_{j+1})$$

Начальное приближение $y^0_{j+1} = y_j$

В качестве точки останова выберем $|y_{j+1}^k - y_j^{k-1}| < \varepsilon$, $\varepsilon=10^{-3}$

Порядок ошибки метода - $O(h)$

In [2]:
eps1 = 10**(-3)
h = 0.1
y_1 = [u_0]

for j in range(num_nodes-1):
    y_prev = y_1[j]
    y_next = y_1[j] + h * func(x[j+1], y_prev)
    while(np.abs(y_prev - y_next) > eps1):
        y_prev = y_next
        y_next = y_1[j] + h * func(x[j+1], y_prev)
    
    y_1.append(y_next)

df = pd.DataFrame({'x': x, 'Euler': y_1})

### 2. Явный метод Рунге-Кутта 2-го порядка с $\alpha=1$

$\alpha = 1$

$$\begin{gathered} y_{j+1}=y_j + \frac{h}{2} (k_1 + k_2) \\ k_1 = f(x_j; y_j) \\ k_2 = f(x_j+h; y_j + hk_1) \end{gathered}$$

Аналог формулы трапеций. Также в литературе называется методом Хойна.

Порядок ошибки метода - $O(h^2)$

In [3]:
y_2 = [u_0]
for j in range(num_nodes-1):
    k_1 = func(x[j], y_2[j])
    k_2 = func(x[j] + h, y_2[j] + h * k_1)
    y_new = y_2[j] + h / 2 * (k_1 + k_2)
    y_2.append(y_new)

df['Runge-Kutta'] = y_2

### 3. Неявный метод Адамса 2-го порядка с реализацией по методу простых итераций

$$y_{j+1} = y_j + \frac{h}{2}(f_j + f_{j+1})$$

В качестве начального приближения $y_{j+1}^0 = y_j$.

В качестве точки останова выберем $|y_{j+1} - y_j| < \varepsilon$, $\varepsilon=10^{-5}$.

Порядок ошибки метода - $O(h^2)$.

In [4]:
eps2 = 10**(-5)
y_3 = [u_0]

for j in range(num_nodes-1):
    func_j = func(x[j], y_3[j])
    y_prev = y_3[j]
    y_next = y_3[j] + h / 2 * (func_j + func(x[j+1], y_prev))
    while(np.abs(y_next - y_prev) > eps2):
        y_prev = y_next
        y_next = y_3[j] + h / 2 * (func_j + func(x[j+1], y_prev))
    y_3.append(y_next)

df['Adams'] = y_3

### 4. Аналитическое решение

Уравнение сводится к линейному обыкновенному уравнению при помощи интегрирующего множителя $\mu(x) = \frac{1}{x}$:

$$\begin{gathered} (\frac{u}{x})' = 0.35e^x - 0.65 \sin x \\ u(x) = x(0.35e^x + 0.65\cos x) \end{gathered}$$

In [5]:
def sol(x):
    return x * (0.35 * np.exp(x) + 0.65 * np.cos(x))

y_true = sol(x)
df['Analytical'] = y_true

### 5. Таблицы решений и погрешностей

In [6]:
print('Table of solutions:')
df

Table of solutions:


,x,Euler,Runge-Kutta,Adams,Analytical
0,0.35,0.387543,0.387543,0.387543,0.387543
1,0.45,0.513602,0.510115,0.510397,0.510390
2,0.55,0.645532,0.637891,0.638480,0.638429
3,0.65,0.784108,0.771345,0.772273,0.772131
4,0.75,0.930472,0.911389,0.912700,0.912411
5,0.85,1.086348,1.059441,1.061189,1.060686
6,0.95,1.254085,1.217481,1.219729,1.218938
7,1.05,1.436718,1.388121,1.390942,1.389779
8,1.15,1.638034,1.574668,1.578145,1.576517
9,1.25,1.862633,1.781193,1.785420,1.783224


In [7]:
df_error = pd.DataFrame()
df_error['x'] = df['x']
df_error['Euler'] = np.abs(df['Analytical'] - df['Euler'])
df_error['Runge-Kutta'] = np.abs(df['Analytical'] - df['Runge-Kutta'])
df_error['Adams'] = np.abs(df['Analytical'] - df['Adams'])

print('Table of errors:')
df_error

Table of errors:


,x,Euler,Runge-Kutta,Adams
0,0.35,0.000000,0.000000,0.000000
1,0.45,0.003212,0.000275,0.000007
2,0.55,0.007104,0.000538,0.000051
3,0.65,0.011977,0.000786,0.000142
4,0.75,0.018061,0.001022,0.000290
5,0.85,0.025662,0.001245,0.000503
6,0.95,0.035147,0.001457,0.000791
7,1.05,0.046939,0.001658,0.001163
8,1.15,0.061517,0.001849,0.001628
9,1.25,0.079409,0.002031,0.002195


### Вывод

Полученные ошибки полностью согласуются с теорией. Порядок ошибки метода Эйлера - $10^{-1}$, методов Рунге-Кутта и Адамса - $10^{-3}$, причём на первой части отрезка метод Адамса даёт лучшие результаты, а в последних двух узлах - метод Рунге-Кутта оказался точнее.

Таким образом,
- реализованы требуемые методы 1-го и 2-го порядков точности,
- получены таблицы приближённых решений и их погрешностей,
- погрешности подтверждают теоретические порядки точности методов.

### Доп: метод Рунге-Кутта 4-го порядка

$$\begin{gathered} y_{j+1}=y_j + \frac{h}{6}(k_1+2k_2+2k_3+k_4) \\ k_1=f(x_j, y_j) \\ k_2=f(x_j + \frac{h}{2}; y_j + \frac{h}{2}k_1) \\ k_3 = f(x_j + \frac{h}{2}; y_j + \frac{h}{2}k_2) \\ k_4 = f(x_j + h; y_j + hk_3) \end{gathered}$$

In [8]:
y_4 = [u_0]
for j in range(num_nodes-1):
    k_1 = func(x[j], y_4[j])
    k_2 = func(x[j] + h/2, y_4[j] + h/2 * k_1)
    k_3 = func(x[j] + h/2, y_4[j] + h/2 * k_2)
    k_4 = func(x[j] + h, y_4[j] + h * k_3)
    y_new = y_4[j] + h/6 * (k_1 + 2*k_2 + 2*k_3 + k_4)
    y_4.append(y_new)

df['Runge-Kutta-4'] = y_4

In [ ]:
print('Table of solutions:')
df

,x,Euler,Runge-Kutta,Adams,Analytical,Runge-Kutta-4
0,0.35,0.387543,0.387543,0.387543,0.387543,0.387543
1,0.45,0.513602,0.510115,0.510397,0.510390,0.510388
2,0.55,0.645532,0.637891,0.638480,0.638429,0.638426
3,0.65,0.784108,0.771345,0.772273,0.772131,0.772127
4,0.75,0.930472,0.911389,0.912700,0.912411,0.912406
5,0.85,1.086348,1.059441,1.061189,1.060686,1.060680
6,0.95,1.254085,1.217481,1.219729,1.218938,1.218932
7,1.05,1.436718,1.388121,1.390942,1.389779,1.389772
8,1.15,1.638034,1.574668,1.578145,1.576517,1.576509
9,1.25,1.862633,1.781193,1.785420,1.783224,1.783216


In [ ]:
print('Table of errors:')
df_error['Runge-Kutta-4'] = np.abs(df['Analytical'] - df['Runge-Kutta-4'])
df_error

,x,Euler,Runge-Kutta,Adams,Runge-Kutta-4
0,0.35,0.000000,0.000000,0.000000,0.000000
1,0.45,0.003212,0.000275,0.000007,0.000002
2,0.55,0.007104,0.000538,0.000051,0.000003
3,0.65,0.011977,0.000786,0.000142,0.000004
4,0.75,0.018061,0.001022,0.000290,0.000005
5,0.85,0.025662,0.001245,0.000503,0.000005
6,0.95,0.035147,0.001457,0.000791,0.000006
7,1.05,0.046939,0.001658,0.001163,0.000007
8,1.15,0.061517,0.001849,0.001628,0.000008
9,1.25,0.079409,0.002031,0.002195,0.000009
